In [9]:
import os
import pandas as pd
import kagglehub
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier


path = kagglehub.dataset_download(
    "eplfixturestoday/premier-league-team-all-time-table-19922026"
)
csv_path = os.path.join(path, "datasets", "team_season_records.csv")
df = pd.read_csv(csv_path)

features = [
    "played", "won", "drawn", "lost",
    "goals_for", "goals_against", "goal_difference",
]


print("NaNs in target:", df["relegated"].isna().sum())
df = df.dropna(subset=features + ["relegated"])

x = df[features]
y = df["relegated"].astype(int)


x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42, stratify=y
)

print("Training features:", x_train.shape)
print("Testing features:", x_test.shape)
print("\nTraining target distribution:\n", y_train.value_counts())
print("\nTesting target distribution:\n", y_test.value_counts())


log_reg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
tree = DecisionTreeClassifier(max_depth=3, random_state=42)
base = DummyClassifier(strategy="most_frequent")

for m in (log_reg, tree, base):
    m.fit(x_train, y_train)


sample = x_test.iloc[:5]
for name, m in [("LogReg", log_reg), ("Tree", tree)]:
    out = pd.DataFrame(
        m.predict_proba(sample),
        columns=["P(stayed)", "P(relegated)"],
        index=sample.index,
    ).round(3)
    out["predicted"] = m.predict(sample)
    out["actual"] = y_test.iloc[:5].values
    print(f"\n{name}\n", out)

print("\nAccuracy")
print("LogReg  :", round(log_reg.score(x_test, y_test), 3))
print("Tree    :", round(tree.score(x_test, y_test), 3))
print("Baseline:", round(base.score(x_test, y_test), 3)) 


print("\nLogReg coefficients (+ = pushes toward relegation):")
print(pd.Series(log_reg[-1].coef_[0], index=features).sort_values())



#==========Model Selection===================#
'''
i would pick Logistic regression as ny model becouse it has the highest test accuracy


'''

rows = []
for name, m in [("LogReg", log_reg), ("Tree", tree)]:
    p = m.predict(x_test)
    rows = []
    for name, m in [("LogReg", log_reg), ("Tree", tree)]:
        p = m.predict(x_test)
        from sklearn.metrics import (
            accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
        )
    print(pd.DataFrame(rows).round(3).to_string(index=False))
print(pd.DataFrame(rows).round(3).to_string(index=False))

for name, m in [("LogReg", log_reg), ("Tree", tree)]:
    cm = confusion_matrix(y_test, m.predict(x_test))
    print(f"\n{name} confusion matrix (rows=actual, cols=predicted)\n", cm)
    tn, fp, fn, tp = cm.ravel()
    print(f"TN={tn} FP={fp} FN={fn} TP={tp}")


skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for name, m in [("LogReg", log_reg), ("Tree", tree)]:
    f1_cv = cross_val_score(m, x, y, cv=skf, scoring="f1")
    print(f"{name} CV F1:", f1_cv.round(3), f"mean={f1_cv.mean():.3f} std={f1_cv.std():.3f}")

NaNs in target: 0
Training features: (548, 7)
Testing features: (138, 7)

Training target distribution:
 relegated
0    466
1     82
Name: count, dtype: int64

Testing target distribution:
 relegated
0    117
1     21
Name: count, dtype: int64

LogReg
      P(stayed)  P(relegated)  predicted  actual
527      1.000         0.000          0       0
537      0.998         0.002          0       0
490      1.000         0.000          0       0
220      0.998         0.002          0       0
565      0.001         0.999          1       1

Tree
      P(stayed)  P(relegated)  predicted  actual
527      0.995         0.005          0       0
537      0.995         0.005          0       0
490      0.995         0.005          0       0
220      0.995         0.005          0       0
565      0.000         1.000          1       1

Accuracy
LogReg  : 0.978
Tree    : 0.942
Baseline: 0.848

LogReg coefficients (+ = pushes toward relegation):
won               -2.996973
goals_for         -0.7611